# SHBT Simulator — Jupyter / Colab Example

This notebook shows how to run custom SHBT simulations from Python and plot the results.

**Prerequisites** (in this environment or on Colab):
- The compiled `shbt_simulator` Rust extension on the Python path.
- Optional: `matplotlib`, `h5py`, `pandas` (see `requirements.txt`).

In [ ]:
import sys, json, os
from pathlib import Path

# If running from the repo examples/ directory, add the repo root to the path.
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import shbt_simulate

# Optional plotting setup
try:
    import matplotlib.pyplot as plt
except Exception as e:
    print('matplotlib not available:', e)

In [ ]:
# Run a custom simulation. You can change branch, observer parameters, etc.
config = {
    'mode': 'all',
    'branch': (26, 8, 312),
    'observer_radius_fraction': 0.125,
    'redshift_max': 3.0,
    'redshift_samples': 9,
    'particles': 512,
}

result = shbt_simulate.simulate(config)
print('branch:', result['config']['branch'])
print('eta_b:', result['audit']['eta_b'])
print('stress_energy_preserved:', result['audit']['stress_energy_preserved'])
print('metric slices:', len(result['audit']['metric_slices']))
print('history entries:', len(result['audit']['history_entries']))

In [ ]:
# Export the full result to JSON or CSV/HDF5
shbt_simulate.export_result(result, '/tmp/shbt_result.json', fmt='json')
print('Exported to /tmp/shbt_result.json')

In [ ]:
# Plot metric eigenvalues across redshift / tau
slices = result['audit']['metric_slices']
taus = [s['tau'] for s in slices]

fig, ax = plt.subplots(figsize=(6, 4))
for dim in range(4):
    vals = [s['eigenvalues'][dim] for s in slices]
    ax.plot(taus, vals, marker='o', label=f'λ{dim}')
ax.set_xlabel('τ')
ax.set_ylabel('Eigenvalue')
ax.set_title('Metric eigenvalues across bulk slices')
ax.legend()
plt.show()

In [ ]:
# Heatmap of the first spatial-metric slice
import numpy as np

sm = np.array(result['audit']['metric_slices'][0]['spatial_metric'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sm, cmap='viridis')
ax.set_title('Spatial metric components (first slice)')
ax.set_xlabel('j')
ax.set_ylabel('i')
fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
# Parameter sweep example
sweep_config = {
    'redshift_samples': [5, 9, 13],
    'observer_radius_fraction': [0.1, 0.125]
}

sweep_result = shbt_simulate.run_sweep(sweep_config)
print('sweep runs:', len(sweep_result['sweep_results']))

# Print eta_b for each sweep configuration
for cfg, res in zip(sweep_result['sweep_configs'], sweep_result['sweep_results']):
    print(cfg, '=> eta_b =', res['audit']['eta_b'])